# BERTopic on a small Bloomberg sample — offline vs online

**Goal:** a first, fast playground for *unsupervised theme discovery* on Bloomberg
headlines. We deliberately work on a **tiny** corpus (≈100 headlines/year × 10
years ≈ 1,000 docs) so every step runs in seconds and we can build intuition for:

1. **Offline BERTopic** — fit once on the whole corpus (embed → UMAP → HDBSCAN → c-TF-IDF).
2. **Online / incremental BERTopic** — `partial_fit` year-by-year with `IncrementalPCA`
   + `MiniBatchKMeans` + `OnlineCountVectorizer`, the way you'd run on a stream.

> **Data:** raw news live in the sibling *Theme-Detection* project and are reached
> through the symlink `data/raw/bloomberg/` — nothing heavy is copied here.
> The small sample is built once by `scripts/build_sample.py` and cached to parquet.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PER_YEAR = 10000
SAMPLE_PATH = PROJECT_ROOT / "data" / "processed" / f"sample_{PER_YEAR}_per_year.parquet"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # small, fast, general-purpose (384-D)
RANDOM_SEED = 42

DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("device:", DEVICE)

device: mps


## 1. Load the cached sample

If the parquet doesn't exist yet, build it once (≈30s for all 10 years). `.xz` has no
random access and the files are ~19 M headlines/year, so to stay fast and within a
small RAM budget the builder decompresses only a **prefix chunk** of each year
(≈ first 10 days) and samples from it. Tradeoff: the sample is drawn from the *start*
of each year, not spread evenly across all 12 months — fine for a first playground.

```bash
uv run python scripts/build_sample.py --per-year 100
```

In [2]:
assert SAMPLE_PATH.exists(), (
    f"{SAMPLE_PATH} missing — run:\n"
    f"    uv run python scripts/build_sample.py --per-year {PER_YEAR}"
)

corpus = pd.read_parquet(SAMPLE_PATH)
# Drop exact-duplicate headlines (wires repost the same story); keep first occurrence.
corpus = corpus.drop_duplicates(subset="Headline").reset_index(drop=True)

docs = corpus["Headline"].tolist()
timestamps = corpus["CaptureTime"].tolist()
years = corpus["year"].tolist()

print(f"{len(docs):,} unique headlines, {corpus['year'].nunique()} years")
display(corpus.groupby("year").size().rename("headlines").to_frame().T)
corpus.head()

99,950 unique headlines, 10 years


year,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
headlines,10000,9999,9995,9997,9991,9995,9993,9994,9993,9993


,CaptureTime,Headline,DerivedTopicsId,WireName,year
0,2016-01-01 00:00:08.985000+00:00,EON: E.ON UK availability update - 31/12/2015,SRCRANK5;NRG;CMDKEY;REMIT;EUROPE;WORLD;SPREGIO...,CO3,2016
1,2016-01-01 00:00:40.077000+00:00,ProactivInvst UK: Gold heads into New Year wit...,METALKEY;CMDKEY;NASIA;EM;PCS;SRCRANK2;WORLD;GL...,NS3,2016
2,2016-01-01 00:00:43.894000+00:00,Mbac Fertilizer: MBAC Provides Update on Its S...,COEVNT;CMDKEY;CHM;WORLD;CACT;BIZNEWS;BUSINESS;...,CO2,2016
3,2016-01-01 00:01:01.497000+00:00,Centamin Plc: 21 Mar 2016 Annual results for t...,BIZNEWS;BUSINESS;ERN;MISC;COS;SRCRANK3;EQUITYKEY,CO6,2016
4,2016-01-01 00:01:32.300000+00:00,Provident Fin: 2016 Q1 interim management stat...,SRCRANK5;MISC;EQUITYKEY,CO3,2016


## 2. Embed the headlines once

Both the offline and online runs reuse the same sentence embeddings, so we compute
them a single time. `all-MiniLM-L6-v2` is a small 384-D general-purpose model — fine
for a first look; swap in a finance-tuned model later to compare.

In [3]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
embeddings = embedder.encode(docs, batch_size=64, show_progress_bar=True)
# float64: the online path (IncrementalPCA → MiniBatchKMeans) needs doubles; the
# offline path is happy with either. Tiny array, so the cast is essentially free.
embeddings = np.asarray(embeddings, dtype="float64")
print("embeddings:", embeddings.shape, embeddings.dtype)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1562 [00:00<?, ?it/s]

embeddings: (99950, 384) float64


## 3. Offline BERTopic

Fit the full pipeline once. With only ~1,000 docs the default HDBSCAN settings would
mark almost everything as an outlier, so we shrink the neighbourhood / cluster sizes
to small-corpus-friendly values:

- `UMAP(n_neighbors=15, n_components=5)` — reduce embeddings before clustering.
- `HDBSCAN(min_cluster_size=10)` — a "topic" needs ≥10 headlines.
- `CountVectorizer(stop_words="english", ngram_range=(1,2))` — drives the c-TF-IDF
  keywords that *describe* each topic.

Topic `-1` is the outlier bucket.

In [4]:
umap_model = UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=RANDOM_SEED
)
hdbscan_model = HDBSCAN(
    min_cluster_size=10, min_samples=5, metric="euclidean",
    cluster_selection_method="eom", prediction_data=True,
)
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2))

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True,
)
topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)

info = topic_model.get_topic_info()
print(f"{(info['Topic'] != -1).sum()} topics + outliers; "
      f"{info.loc[info['Topic'] == -1, 'Count'].sum()} / {len(docs)} docs are outliers")
info.head(20)

2026-06-17 11:19:14,421 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


2026-06-17 11:20:20,673 - BERTopic - Dimensionality - Completed ✓


2026-06-17 11:20:20,758 - BERTopic - Cluster - Start clustering the reduced embeddings


2026-06-17 11:20:28,521 - BERTopic - Cluster - Completed ✓


2026-06-17 11:20:28,559 - BERTopic - Representation - Fine-tuning topics using representation models.


2026-06-17 11:20:32,715 - BERTopic - Representation - Completed ✓


1439 topics + outliers; 42551 / 99950 docs are outliers


,Topic,Count,Name,Representation,Representative_Docs
0,-1,42551,-1_new_news_announces_china,"[new, news, announces, china, year, business, ...",[ABC Online (AU): Wall St drops on Chinese eco...
1,0,2666,0_01 02_01 01_12 31_02,"[01 02, 01 01, 12 31, 02, 01, 12, 2019 01, 202...","[Burke Richard T: 4 2018/12/31, King Richard H..."
2,1,852,1_bowl_basketball_penn state_penn,"[bowl, basketball, penn state, penn, sugar bow...",[USA Today: Alex's best bet: College Football ...
3,2,826,2_bitcoin_crypto_solana_ethereum,"[bitcoin, crypto, solana, ethereum, btc, bitco...",[Crypto News: Best Cryptocurrencies to Invest ...
4,3,566,3_food_foods_meat_mcdonald,"[food, foods, meat, mcdonald, burger, restaura...",[Street Insider: McDonald's expands Beyond Mea...
5,4,538,4_2025 01_2017 01_corp_01,"[2025 01, 2017 01, corp, 01, 01 04, 01 03, 01 ...","[Workhorse Group Inc.: 4 2023-01-04, SAPIENS I..."
6,5,506,5_covid_cases_covid 19_coronavirus,"[covid, cases, covid 19, coronavirus, 19, 19 c...","[Star-Ledger: N.J. reports 5,528 new COVID-19 ..."
7,6,494,6_vaccine_19 vaccine_covid 19_covid,"[vaccine, 19 vaccine, covid 19, covid, moderna...",[ArkansasDemocrat: EU agency approves Moderna'...
8,7,493,7_bayern_liverpool_utd_man utd,"[bayern, liverpool, utd, man utd, chelsea, tra...",[Tribal Football: Bayern Munich delighted as H...
9,8,486,8_potash_agrium_potash corp_fertilizer,"[potash, agrium, potash corp, fertilizer, pota...",[Saskatchewan Opposes BHP’s Bid for Potash Cor...


### Inspect a few topics — top keywords and representative headlines

In [5]:
for topic_id in info.loc[info["Topic"] != -1, "Topic"].head(6):
    words = ", ".join(w for w, _ in topic_model.get_topic(topic_id)[:8])
    print(f"\n── Topic {topic_id} ── {words}")
    for doc in topic_model.get_representative_docs(topic_id)[:3]:
        print("   •", doc[:120])


── Topic 0 ── 01 02, 01 01, 12 31, 02, 01, 12, 2019 01, 2023 01
   • Burke Richard T: 4 2018/12/31
   • King Richard H: 4 2017/12/29
   • Hill John W: 4 2019/12/31

── Topic 1 ── bowl, basketball, penn state, penn, sugar bowl, alabama, football, state
   • USA Today: Alex's best bet: College Football Playoff – Notre Dame vs. Penn State prediction
   • Mpls Star-Trib: Defense and special teams lift Notre Dame to 23-10 win over Georgia in Sugar Bowl CFP quarterfinal
   • Ariz Republic: Penn State beats Boise State in Fiesta Bowl to make College Football Playoff semifinals

── Topic 2 ── bitcoin, crypto, solana, ethereum, btc, bitcoin price, dogecoin, daily bitcoin
   • Crypto News: Best Cryptocurrencies to Invest in for January 2025 Ripple (XRP), Cardano (ADA), and Solana (SOL) Amid Ligh
   • Daily Bitcoin: Who Created Bitcoin? | Bitcoin news
   • Scotsman: Bitcoin price: Why is crypto crashing? Crypto prices of BTC, Ethereum, Dogecoin, Solana and more in 2022 crash

── Topic 3 ── food,

### Visualise

`visualize_barchart` shows the defining keywords per topic; `visualize_topics_over_time`
uses the `CaptureTime` stamps to show how each theme's headline volume moves across the
10 years — a first hint at *emerging* vs *fading* themes.

In [6]:
topic_model.visualize_barchart(top_n_topics=8, n_words=8)

In [7]:
topics_over_time = topic_model.topics_over_time(docs, timestamps, nr_bins=20)
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=6)

0it [00:00, ?it/s]

1it [00:00,  1.09it/s]

2it [00:37, 21.95s/it]

3it [01:13, 28.18s/it]

4it [01:47, 30.75s/it]

...

7it [03:39, 35.71s/it]

8it [04:28, 39.95s/it]

9it [05:20, 43.59s/it]

10it [06:00, 42.54s/it]

10it [06:00, 36.05s/it]

## 4. Online / incremental BERTopic

On a real feed you can't refit on the whole history every day. The online recipe
swaps the three stateful stages for incremental ones and calls `partial_fit` on each
batch — here, one batch **per year**:

| stage        | offline                | online (incremental)        |
|--------------|------------------------|-----------------------------|
| dim. reduce  | `UMAP`                 | `IncrementalPCA`            |
| cluster      | `HDBSCAN`              | `MiniBatchKMeans` (fixed k) |
| keywords     | `CountVectorizer`      | `OnlineCountVectorizer`     |

Trade-off: you must fix the number of clusters up front (`MiniBatchKMeans`), and the
keyword vocabulary updates with a `decay` so stale terms fade. This is the path to
explore for *streaming* theme discovery.

In [8]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import IncrementalPCA
from bertopic.vectorizers import OnlineCountVectorizer

online_umap = IncrementalPCA(n_components=5)
online_cluster = MiniBatchKMeans(n_clusters=10, random_state=RANDOM_SEED)
online_vectorizer = OnlineCountVectorizer(stop_words="english", decay=0.01)

online_model = BERTopic(
    umap_model=online_umap,
    hdbscan_model=online_cluster,
    vectorizer_model=online_vectorizer,
    verbose=False,
)

# Feed one year at a time, in chronological order — simulating a stream.
for year in sorted(corpus["year"].unique()):
    mask = corpus["year"].values == year
    online_model.partial_fit(
        [d for d, m in zip(docs, mask) if m],
        embeddings=embeddings[mask],
    )
    n_topics = len(set(online_model.topics_)) - (1 if -1 in online_model.topics_ else 0)
    print(f"after {year}: {n_topics} topics, {len(online_model.topics_):,} docs seen")

online_model.get_topic_info().head(12)

after 2016: 10 topics, 10,000 docs seen


after 2017: 10 topics, 9,999 docs seen


after 2018: 10 topics, 9,995 docs seen


after 2019: 10 topics, 9,997 docs seen


after 2020: 10 topics, 9,991 docs seen


...

after 2022: 10 topics, 9,993 docs seen


after 2023: 10 topics, 9,994 docs seen


after 2024: 10 topics, 9,993 docs seen


after 2025: 10 topics, 9,993 docs seen


,Topic,Count,Name,Representation,Representative_Docs
0,0,9097,0_news_trump_meta_new,"[news, trump, meta, new, china, musk, ai, cybe...",NaN
1,1,11716,1_limited_8k_india_nse,"[limited, 8k, india, nse, corp, 2025, 144, ann...",NaN
2,2,13954,2_new_orleans_news_trump,"[new, orleans, news, trump, attack, state, jim...",NaN
3,3,11506,3_price_market_stocks_shares,"[price, market, stocks, shares, stock, 2025, t...",NaN
4,4,8300,4_shares_insider_investors_ceo,"[shares, insider, investors, ceo, street, anno...",NaN
5,5,8238,5_20250102_20241231_2025_20250101,"[20250102, 20241231, 2025, 20250101, jan, 2024...",NaN
6,6,10829,6_2025_ai_india_new,"[2025, ai, india, new, times, ces, news, says,...",NaN
7,7,8822,7_new_2025_vs_news,"[new, 2025, vs, news, nfl, playoff, sports, st...",NaN
8,8,7714,8_filed_fec_ferc_gas,"[filed, fec, ferc, gas, bank, f3xn, tax, submi...",NaN
9,9,9774,9_india_2025_bank_2024,"[india, 2025, bank, 2024, limited, nse, rs, tr...",NaN


> **Note** `partial_fit` keeps `online_model.topics_` for the *most recent* batch only.
> To track topics across the whole stream, accumulate the returned topics yourself, or
> look at `BERTopic.merge_models` for combining independently-fit models incrementally.

## 5. Where to go next

- **Compare embedders:** finance-tuned (e.g. `FinLang/finance-embeddings-investopedia`)
  vs MiniLM on the same sample.
- **Scale up gradually:** 100 → 1,000 → 10,000 per year (`--per-year`) and watch how
  topic granularity and outlier rate change.
- **Tune granularity:** `min_cluster_size` / `nr_topics` (offline) and `n_clusters`
  (online) control how broad vs fine the themes are.
- **Emergence:** lean on `topics_over_time` to flag themes whose volume accelerates —
  the bridge to the `bertrend`-style experiments in the `0.x` notebooks.